# Tutorial: LunaSeis-1 Apollo Waveform Inference

**Author:** Advaith Praveen (APRK) · Archeum Studios

**Audience:** students and researchers who want to reproduce compact Apollo seismic inference.

**Prerequisites:** Python basics and a local clone of LunaSeis-1 (or a future public GitHub clone in Colab).

**Learning goals:** inspect the selected station-held-out checkpoint, score a waveform, scan MiniSEED, and interpret warnings correctly. This is a research prototype—not a flight-ready detector.


## Outline

1. Import the reproducible inference package
2. Inspect the station-specific model metadata
3. Run a deterministic synthetic smoke test
4. Optionally scan a checksum-verified Apollo MiniSEED file
5. Review pitfalls and an exercise


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np

from lunaseis import LunaSeisDetector

STATION = 'S15'
detector = LunaSeisDetector(STATION)
detector.metadata()


## Step 1 — Deterministic smoke test

The synthetic waveform only verifies preprocessing and inference wiring. It is **not** scientific performance evidence. Valid Apollo counts are combined with explicit `-1` gap sentinels.


In [ ]:
rng = np.random.default_rng(20260714)
synthetic = rng.normal(425, 8, 4096).round()
synthetic[900:920] = -1  # explicit missing samples
result = detector.predict_values(synthetic)
print(json.dumps(result, indent=2))


## Step 2 — Scan a real MiniSEED file (optional)

Set `EXAMPLE_MSEED` to a checksum-verified Apollo MH product reconstructed by the repository download scripts. The scanner uses 600-second windows and a 60-second stride.

### Pitfalls

- A score is not a calibrated probability.
- A trigger is not a confirmed moonquake.
- Use the checkpoint matching the station; the API does this automatically.
- Never replace `-1` gaps with interpolated signal silently.

### Exercise

Count the candidate windows in a real daily file and compare that count with the fraction of valid/scored windows. Explain why neither number is an event-recall metric.


In [ ]:
EXAMPLE_MSEED = Path('path/to/checksum_verified_apollo_mh.mseed')

if EXAMPLE_MSEED.exists():
    windows = detector.scan_mseed(EXAMPLE_MSEED)
    summary = {
        'windows': len(windows),
        'scored': sum(row['score'] is not None for row in windows),
        'candidate_triggers': sum(row['triggered'] is True for row in windows),
    }
    print(summary)
else:
    print('Set EXAMPLE_MSEED after reconstructing an official product. Synthetic smoke test completed successfully.')
